In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt

gpus = tf.config.list_physical_devices('GPU')

vram_limit = 10240

if gpus:
    try:
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit = vram_limit)]
        )
        print(f"VRAM capped at {vram_limit} MB")
    except RuntimeError as e:
        print(e)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Load CIFAR-10: 32x32 RGB images, 10 classes
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)

# Use a smaller subset for a 2-hour lab
X_train_small = X_train[:10000]
y_train_small = y_train[:10000]
X_test_small = X_test[:2000]
y_test_small = y_test[:2000]

print(X_train_small.shape, y_train_small.shape)
print(X_test_small.shape, y_test_small.shape)

VRAM capped at 10240 MB
 46153728/170498071 ━━━━━━━━━━━━━━━━━━━━ 51:56 25us/step

In [ ]:
base_model = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the pre-trained base model
base_model.trainable = False

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Resizing(96, 96)(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_feature = model.fit(
    X_train_small, y_train_small,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

feature_loss, feature_test_acc = model.evaluate(
    X_test_small, y_test_small, verbose=0
)
print("Feature extraction test accuracy:", feature_test_acc)

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Keep most layers frozen; fine-tune only the last 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    X_train_small, y_train_small,
    epochs=2,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

fine_loss, fine_test_acc = model.evaluate(
    X_test_small, y_test_small, verbose=0
)
print("Fine-tuned test accuracy:", fine_test_acc)

In [ ]:
plt.plot(history_feature.history["accuracy"], label="Feature train")
plt.plot(history_feature.history["val_accuracy"], label="Feature validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Feature extraction")
plt.legend()
plt.show()

plt.plot(history_fine.history["accuracy"], label="Fine-tune train")
plt.plot(history_fine.history["val_accuracy"], label="Fine-tune validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fine-tuning")
plt.legend()
plt.show()

In [ ]:
base_model_2 = MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,
    weights="imagenet"
)
base_model_2.trainable = False

inputs_2 = keras.Input(shape=(32, 32, 3))
x2 = layers.Resizing(96, 96)(inputs_2)
x2 = preprocess_input(x2)
x2 = base_model_2(x2, training=False)
x2 = layers.GlobalAveragePooling2D()(x2)
x2 = layers.Dropout(0.2)(x2)
outputs_2 = layers.Dense(10, activation="softmax")(x2)

model_2 = keras.Model(inputs_2, outputs_2)

model_2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_mod_feature = model_2.fit(
    X_train_small, y_train_small,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Fine-tune only the last 10 layers
base_model_2.trainable = True
for layer in base_model_2.layers[:-10]:
    layer.trainable = False

model_2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history_mod_fine = model_2.fit(
    X_train_small, y_train_small,
    epochs=2,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

mod_loss, mod_test_acc = model_2.evaluate(
    X_test_small, y_test_small, verbose=0
)
print("Modified experiment (last 10 layers) test accuracy:", mod_test_acc)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature extraction
axes[0].plot(history_feature.history["accuracy"], label="Train", marker="o")
axes[0].plot(history_feature.history["val_accuracy"], label="Validation", marker="s")
axes[0].set_title("Feature Extraction")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

# Fine-tuning last 20 layers
axes[1].plot(history_fine.history["accuracy"], label="Train", marker="o")
axes[1].plot(history_fine.history["val_accuracy"], label="Validation", marker="s")
axes[1].set_title("Fine-Tuning (Last 20 Layers)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

# Modified experiment last 10 layers
axes[2].plot(history_mod_fine.history["accuracy"], label="Train", marker="o")
axes[2].plot(history_mod_fine.history["val_accuracy"], label="Validation", marker="s")
axes[2].set_title("Modified Last 10 Layers")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Accuracy")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature extraction loss
axes[0].plot(history_feature.history["loss"], label="Train loss", marker="o")
axes[0].plot(history_feature.history["val_loss"], label="Validation loss", marker="s")
axes[0].set_title("Feature Extraction Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Fine-tuning loss
axes[1].plot(history_fine.history["loss"], label="Train loss", marker="o")
axes[1].plot(history_fine.history["val_loss"], label="Validation loss", marker="s")
axes[1].set_title("Fine-Tuning Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history_mod_fine.history["loss"], label="Train loss", marker="o")
axes[2].plot(history_mod_fine.history["val_loss"], label="Validation loss", marker="s")
axes[2].set_title("Modified Last 10 Layers Loss")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Loss")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()